
# Data Transformation and Analysis

## Mounting

In [0]:
if not any(mount.mountPoint == '/mnt/bronze' for mount in dbutils.fs.mounts()):
    dbutils.fs.mount(
    source="wasbs://bronze@secondprojecteva.blob.core.windows.net",
    mount_point="/mnt/bronze",
    extra_configs={"fs.azure.account.key.secondprojecteva.blob.core.windows.net": "key"}
)





## Data Normalization

## Order table normalization and new column calculation

The Orders table contains all the shipping information, while the Customers table holds customer details. These two tables are linked via the CustomerID column. Therefore, let's proceed by removing the last 6 columns.

In [0]:
from pyspark.sql.functions import datediff, col
orders_df = spark.read.parquet("/mnt/bronze/Orders")


all_columns = orders_df.columns

columns_to_drop = all_columns[-6:]


orders_df = orders_df.drop(*columns_to_drop)


display(orders_df)


OrderID,CustomerID,EmployeeID,OrderDate,RequiredDate,ShippedDate,ShipVia,Freight
10248,VINET,5,1996-07-04T00:00:00Z,1996-08-01T00:00:00Z,1996-07-16T00:00:00Z,3,32.3800
10249,TOMSP,6,1996-07-05T00:00:00Z,1996-08-16T00:00:00Z,1996-07-10T00:00:00Z,1,11.6100
10250,HANAR,4,1996-07-08T00:00:00Z,1996-08-05T00:00:00Z,1996-07-12T00:00:00Z,2,65.8300
10251,VICTE,3,1996-07-08T00:00:00Z,1996-08-05T00:00:00Z,1996-07-15T00:00:00Z,1,41.3400
10252,SUPRD,4,1996-07-09T00:00:00Z,1996-08-06T00:00:00Z,1996-07-11T00:00:00Z,2,51.3000
10253,HANAR,3,1996-07-10T00:00:00Z,1996-07-24T00:00:00Z,1996-07-16T00:00:00Z,2,58.1700
10254,CHOPS,5,1996-07-11T00:00:00Z,1996-08-08T00:00:00Z,1996-07-23T00:00:00Z,2,22.9800
10255,RICSU,9,1996-07-12T00:00:00Z,1996-08-09T00:00:00Z,1996-07-15T00:00:00Z,3,148.3300
10256,WELLI,3,1996-07-15T00:00:00Z,1996-08-12T00:00:00Z,1996-07-17T00:00:00Z,2,13.9700
10257,HILAA,4,1996-07-16T00:00:00Z,1996-08-13T00:00:00Z,1996-07-22T00:00:00Z,3,81.9100


Add new column for Delivery days

In [0]:
orders_df = orders_df.withColumn("DeliveryDays", datediff(col("ShippedDate"), col("OrderDate")))
display(orders_df)

OrderID,CustomerID,EmployeeID,OrderDate,RequiredDate,ShippedDate,ShipVia,Freight,DeliveryDays
10248,VINET,5,1996-07-04T00:00:00Z,1996-08-01T00:00:00Z,1996-07-16T00:00:00Z,3,32.3800,12
10249,TOMSP,6,1996-07-05T00:00:00Z,1996-08-16T00:00:00Z,1996-07-10T00:00:00Z,1,11.6100,5
10250,HANAR,4,1996-07-08T00:00:00Z,1996-08-05T00:00:00Z,1996-07-12T00:00:00Z,2,65.8300,4
10251,VICTE,3,1996-07-08T00:00:00Z,1996-08-05T00:00:00Z,1996-07-15T00:00:00Z,1,41.3400,7
10252,SUPRD,4,1996-07-09T00:00:00Z,1996-08-06T00:00:00Z,1996-07-11T00:00:00Z,2,51.3000,2
10253,HANAR,3,1996-07-10T00:00:00Z,1996-07-24T00:00:00Z,1996-07-16T00:00:00Z,2,58.1700,6
10254,CHOPS,5,1996-07-11T00:00:00Z,1996-08-08T00:00:00Z,1996-07-23T00:00:00Z,2,22.9800,12
10255,RICSU,9,1996-07-12T00:00:00Z,1996-08-09T00:00:00Z,1996-07-15T00:00:00Z,3,148.3300,3
10256,WELLI,3,1996-07-15T00:00:00Z,1996-08-12T00:00:00Z,1996-07-17T00:00:00Z,2,13.9700,2
10257,HILAA,4,1996-07-16T00:00:00Z,1996-08-13T00:00:00Z,1996-07-22T00:00:00Z,3,81.9100,6


## Product Table and Creation of Categry table

We are designing a Categories table to normalize the products_df dataset. The CategoryID column will act as the primary key for establishing relational integrity.

In [0]:
from pyspark.sql import functions as F


products_df = spark.read.parquet("/mnt/bronze/Products_denorm")


all_columns_prod = products_df.columns


columns_to_drop = all_columns_prod[-3:]


#Create Categories DataFrame
categories_df = products_df.select(*columns_to_drop)
categories_df=categories_df.dropDuplicates()
categories_df=categories_df.sort(F.col("Category_ID").asc())




display(categories_df)


Category_ID,CategoryName,Description
1,Beverages,"Soft drinks, coffees, teas, beers, and ales"
2,Condiments,"Sweet and savory sauces, relishes, spreads, and seasonings"
3,Confections,"Desserts, candies, and sweet breads"
4,Dairy Products,Cheeses
5,Grains/Cereals,"Breads, crackers, pasta, and cereal"
6,Meat/Poultry,Prepared meats
7,Produce,Dried fruit and bean curd
8,Seafood,Seaweed and fish


Remove the Categoriy related columns from the product data frame and also eliminate the 'SupplierName' column, as the 'SupplierID' is already available.

In [0]:
all_columns_prod = products_df.columns
columns_to_drop_fromprod = all_columns_prod[-4:]
products_df = products_df.drop(*columns_to_drop_fromprod)

display(products_df)

ProductID,ProductName,SupplierID,CategoryID,QuantityPerUnit,UnitPrice,UnitsInStock,UnitsOnOrder,ReorderLevel,Discontinued
1,Chai,1,1,10 boxes x 20 bags,18.0000,39,0,10,false
2,Chang,1,1,24 - 12 oz bottles,19.0000,17,40,25,false
3,Aniseed Syrup,1,2,12 - 550 ml bottles,10.0000,13,70,25,false
4,Chef Anton's Cajun Seasoning,2,2,48 - 6 oz jars,22.0000,53,0,0,false
5,Chef Anton's Gumbo Mix,2,2,36 boxes,21.3500,0,0,0,true
6,Grandma's Boysenberry Spread,3,2,12 - 8 oz jars,25.0000,120,0,25,false
7,Uncle Bob's Organic Dried Pears,3,7,12 - 1 lb pkgs.,30.0000,15,0,10,false
8,Northwoods Cranberry Sauce,3,2,12 - 12 oz jars,40.0000,6,0,0,false
9,Mishi Kobe Niku,4,6,18 - 500 g pkgs.,97.0000,29,0,0,true
10,Ikura,4,8,12 - 200 ml jars,31.0000,31,0,0,false


## Creating Gender Column for Employees

In [0]:
from pyspark.sql import functions as F

employee_df = spark.read.parquet("/mnt/bronze/Employees")
employee_df = employee_df.withColumn(
    "gender",
    F.when(F.col("TitleOfCourtesy") == "Ms.", "Female")
    .when(F.col("TitleOfCourtesy") == "Mrs.", "Female")
    .when(F.col("TitleOfCourtesy") == "Mr.", "Male")
    .otherwise("Other")  
)


display(employee_df)


EmployeeID,LastName,FirstName,Title,TitleOfCourtesy,BirthDate,HireDate,Address,City,Region,PostalCode,Country,HomePhone,Extension,Photo,Notes,ReportsTo,PhotoPath,gender
1,Davolio,Nancy,Sales Representative,Ms.,1948-12-08T00:00:00Z,1992-05-01T00:00:00Z,507 - 20th Ave. E.Apt. 2A,Seattle,WA,98122,USA,(206) 555-9857,5467,FRwvAAIAAAANAA4AFAAhAP////9CaXRtYXAgSW1hZ2UAUGFpbnQuUGljdHVyZQABBQAAAgAAAAcAAABQQnJ1c2gAAAAAAAAAAAAgVAAAQk0gVAAAAAAAAHYAAAAoAAAAwAAAAN8AAAABAAQAAAAAAKBTAADODgAA2A4AAAAAAAA= (truncated),"Education includes a BA in psychology from Colorado State University in 1970. She also completed ""The Art of the Cold Call."" Nancy is a member of Toastmasters International.",2,http://accweb/emmployees/davolio.bmp,Female
2,Fuller,Andrew,"Vice President, Sales",Dr.,1952-02-19T00:00:00Z,1992-08-14T00:00:00Z,908 W. Capital Way,Tacoma,WA,98401,USA,(206) 555-9482,3457,FRwvAAIAAAANAA4AFAAhAP////9CaXRtYXAgSW1hZ2UAUGFpbnQuUGljdHVyZQABBQAAAgAAAAcAAABQQnJ1c2gAAAAAAAAAAAAgVAAAQk0gVAAAAAAAAHYAAAAoAAAAwAAAAN8AAAABAAQAAAAAAKBTAADODgAA2A4AAAAAAAA= (truncated),"Andrew received his BTS commercial in 1974 and a Ph.D. in international marketing from the University of Dallas in 1981. He is fluent in French and Italian and reads German. He joined the company as a sales representative, was promoted to sales manager in January 1992 and to vice president of sales in March 1993. Andrew is a member of the Sales Management Roundtable, the Seattle Chamber of Commerce, and the Pacific Rim Importers Association.",null,http://accweb/emmployees/fuller.bmp,Other
3,Leverling,Janet,Sales Representative,Ms.,1963-08-30T00:00:00Z,1992-04-01T00:00:00Z,722 Moss Bay Blvd.,Kirkland,WA,98033,USA,(206) 555-3412,3355,FRwvAAIAAAANAA4AFAAhAP////9CaXRtYXAgSW1hZ2UAUGFpbnQuUGljdHVyZQABBQAAAgAAAAcAAABQQnJ1c2gAAAAAAAAAAACAVAAAQk2AVAAAAAAAAHYAAAAoAAAAwAAAAOAAAAABAAQAAAAAAABUAADODgAA2A4AAAAAAAA= (truncated),Janet has a BS degree in chemistry from Boston College (1984). She has also completed a certificate program in food retailing management. Janet was hired as a sales associate in 1991 and promoted to sales representative in February 1992.,2,http://accweb/emmployees/leverling.bmp,Female
4,Peacock,Margaret,Sales Representative,Mrs.,1937-09-19T00:00:00Z,1993-05-03T00:00:00Z,4110 Old Redmond Rd.,Redmond,WA,98052,USA,(206) 555-8122,5176,FRwvAAIAAAANAA4AFAAhAP////9CaXRtYXAgSW1hZ2UAUGFpbnQuUGljdHVyZQABBQAAAgAAAAcAAABQQnJ1c2gAAAAAAAAAAAAgVAAAQk0gVAAAAAAAAHYAAAAoAAAAwAAAAN8AAAABAAQAAAAAAKBTAADODgAA2A4AAAAAAAA= (truncated),Margaret holds a BA in English literature from Concordia College (1958) and an MA from the American Institute of Culinary Arts (1966). She was assigned to the London office temporarily from July through November 1992.,2,http://accweb/emmployees/peacock.bmp,Female
5,Buchanan,Steven,Sales Manager,Mr.,1955-03-04T00:00:00Z,1993-10-17T00:00:00Z,14 Garrett Hill,London,null,SW1 8JR,UK,(71) 555-4848,3453,FRwvAAIAAAANAA4AFAAhAP////9CaXRtYXAgSW1hZ2UAUGFpbnQuUGljdHVyZQABBQAAAgAAAAcAAABQQnJ1c2gAAAAAAAAAAAAgVAAAQk0gVAAAAAAAAHYAAAAoAAAAwAAAAN8AAAABAAQAAAAAAKBTAADODgAA2A4AAAAAAAA= (truncated),"Steven Buchanan graduated from St. Andrews University, Scotland, with a BSC degree in 1976. Upon joining the company as a sales representative in 1992, he spent 6 months in an orientation program at the Seattle office and then returned to his permanent post in London. He was promoted to sales manager in March 1993. Mr. Buchanan has completed the courses ""Successful Telemarketing"" and ""International Sales Management."" He is fluent in French.",2,http://accweb/emmployees/buchanan.bmp,Male
6,Suyama,Michael,Sales Representative,Mr.,1963-07-02T00:00:00Z,1993-10-17T00:00:00Z,Coventry House Miner Rd.,London,null,EC2 7JR,UK,(71) 555-7773,428,FRwvAAIAAAANAA4AFAAhAP////9CaXRtYXAgSW1hZ2UAUGFpbnQuUGljdHVyZQABBQAAAgAAAAcAAABQQnJ1c2gAAAAAAAAAAAAgVAAAQk0WVAAAAAAAAHYAAAAoAAAAwAAAAN8AAAABAAQAAAAAAKBTAADODgAA2A4AAAAAAAA= (truncated),"Michael is a graduate of Sussex University (MA, economics, 1983) and the Univer

In [0]:
employee_df.createOrReplaceTempView("Employees")
orders_df.createOrReplaceTempView("Orders")

In [0]:


# Save it to Silver container as DeltaLake format

categories_df.write.format("parquet").mode("overwrite").save("/mnt/silver/Categories")
employee_df.write.format("parquet").mode("overwrite").save("/mnt/silver/Employees")
products_df.write.format("parquet").mode("overwrite").save("/mnt/silver/Products_denorm")
orders_df.write.format("parquet").mode("overwrite").save("/mnt/silver/Orders")







